In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 - Preprocesamiento y Feature Store
# MAGIC
# MAGIC **Control:** control-2
# MAGIC
# MAGIC Entrada:
# MAGIC `workspace.control2.control2_clientes_raw`
# MAGIC
# MAGIC Salida:
# MAGIC `workspace.control2.control2_clientes_features`
# MAGIC
# MAGIC Se generan variables derivadas y se registra la tabla como Feature Table
# MAGIC mediante `FeatureEngineeringClient`.

# COMMAND ----------

from pyspark.sql import functions as F

RAW_TABLE = "workspace.control2.control2_clientes_raw"
FEATURE_TABLE = "workspace.control2.control2_clientes_features"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Leer tabla Delta

# COMMAND ----------

df = spark.table(RAW_TABLE)

print("Registros:", df.count())
display(df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Ingeniería de variables

# COMMAND ----------

df_features = (
    df
    .withColumn(
        "ratio_deuda_ingreso",
        F.when(
            F.col("ingreso_anual_usd") > 0,
            (F.col("deuda_mensual_usd") * 12) / F.col("ingreso_anual_usd")
        ).otherwise(0.0)
    )
    .withColumn(
        "saldo_ingreso_ratio",
        F.when(
            F.col("ingreso_anual_usd") > 0,
            F.col("saldo_promedio_usd") / F.col("ingreso_anual_usd")
        ).otherwise(0.0)
    )
    .withColumn(
        "actividad_cliente",
        F.col("transacciones_mensuales") * F.col("productos_activos")
    )
    .withColumn(
        "antiguedad_cliente_anios",
        F.col("antiguedad_cliente_meses") / F.lit(12.0)
    )
    .withColumn(
        "experiencia_relativa",
        F.when(
            F.col("edad") > 18,
            F.col("experiencia_anios") / (F.col("edad") - 18)
        ).otherwise(0.0)
    )
    .withColumn(
        "segmento_ingreso",
        F.when(F.col("ingreso_anual_usd") < 30000, "Bajo")
         .when(F.col("ingreso_anual_usd") < 50000, "Medio")
         .when(F.col("ingreso_anual_usd") < 80000, "Alto")
         .otherwise("Premium")
    )
    .withColumn(
        "score_alto",
        F.when(F.col("score_crediticio") >= 700, 1).otherwise(0)
    )
    .withColumn(
        "sin_mora",
        F.when(F.col("mora_12m") == 0, 1).otherwise(0)
    )
)

display(df_features.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Seleccionar variables para Feature Store
# MAGIC
# MAGIC `cliente_id` será la clave primaria y `supera_50k` será el target.

# COMMAND ----------

feature_columns = [
    "cliente_id",
    "edad",
    "antiguedad_laboral_anios",
    "experiencia_anios",
    "nivel_educacion",
    "sector_economico",
    "tipo_empleo",
    "ciudad",
    "deuda_mensual_usd",
    "score_crediticio",
    "productos_activos",
    "antiguedad_cliente_meses",
    "transacciones_mensuales",
    "saldo_promedio_usd",
    "mora_12m",
    "ratio_deuda_ingreso",
    "saldo_ingreso_ratio",
    "actividad_cliente",
    "antiguedad_cliente_anios",
    "experiencia_relativa",
    "segmento_ingreso",
    "score_alto",
    "sin_mora",
    "supera_50k"
]

df_feature = df_features.select(*feature_columns)

display(df_feature.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Registrar en Feature Store / Feature Engineering

# COMMAND ----------

from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Intentamos crear la Feature Table.
# Si ya existe, la actualización se realiza mediante write_table cuando está disponible.
try:
    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["cliente_id"],
        df=df_feature,
        description="Features del proyecto control-2 para predecir clientes que superan USD 50K."
    )
    print("Feature Table creada:", FEATURE_TABLE)

except Exception as create_error:
    print("La Feature Table puede existir previamente.")
    print("Detalle:", str(create_error)[:800])

    # Actualización si la API disponible en el runtime soporta write_table.
    try:
        fe.write_table(
            name=FEATURE_TABLE,
            df=df_feature,
            mode="overwrite"
        )
        print("Feature Table actualizada:", FEATURE_TABLE)

    except Exception as write_error:
        print("No fue posible escribir mediante FeatureEngineeringClient.")
        print("Detalle:", str(write_error)[:800])
        print("")
        print("Se crea/actualiza una tabla Delta de respaldo para no perder el pipeline.")

        (
            df_feature.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(FEATURE_TABLE)
        )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Verificación

# COMMAND ----------

display(spark.table(FEATURE_TABLE).limit(10))
print("Feature Table / tabla de features:", FEATURE_TABLE)
